<a href="https://colab.research.google.com/github/aelshehawy/gesis-python-2026/blob/main/Day%203/Sessions/Day3_Session6_Visualization_Finale.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Visualization & the Grand Finale

## Introduction to Python - GESIS Workshop 2026

**Day 3 · Session 6 (Afternoon)** · Thursday, 27 August 2026 · 14:00 - 17:00 (CEST)

**Ashrakat Elshehawy** (Course Convenor, UCL) · a.elshehawy@ucl.ac.uk

**Victor Kreitmann** (Teaching Assistant, UCL) · victor.kreitmann.24@ucl.ac.uk

---

### Our plan for this session

1. Grouping & aggregating with .groupby()
2. Combining datasets with pd.merge
3. Dates: pulling the year out of a date
4. matplotlib: line, bar, histogram, scatter
5. seaborn: statistical plots with style
6. The Grand Finale: animated bubbles, world maps & sunbursts (plotly)
7. Putting it ALL together: a mini-dashboard
8. Wrap-up: your Python journey continues

**How to use this notebook:** run every cell yourself (press the play button or hit `Shift + Enter`), change things, break things, and ask questions at any time - that is exactly what this course is for!

In [95]:
# Setup: our full toolkit for the finale - run this first!
import pandas as pd # you already know pandas
import numpy as np #numpy is a library for numerical computing to do complex matgh
import matplotlib.pyplot as plt      # the classic plotting library ("plt")
import seaborn as sns                # statistical plots with style ("sns")
import plotly.express as px          # interactive & animated plots ("px")



In [ ]:
DATA_URL = "https://raw.githubusercontent.com/aelshehawy/gesis-python-2026/main/data/"
gapminder = pd.read_csv(DATA_URL + "gapminder.csv")

gapminder

In [ ]:
# just 2007
gm2007 = gapminder[gapminder["year"] == 2007]

print("All systems go!", gapminder.shape)

# Grouping & aggregating: `.groupby()`

This morning's exercise pain: computing a mean **per continent** meant one filter + one `.mean()` per continent. Five continents, five code blocks...

`.groupby()` does *split → apply → combine* in **one line**:

1. **Split** the rows into groups (one per continent),
2. **apply** a statistic to each group (mean, count, max...),
3. **combine** the results into a tidy table.

In [ ]:
# Mean life expectancy per continent, 2007 - ONE line:
gm2007.groupby("continent")["lifeExp"].mean()

Read the recipe left to right: *take gm2007 → group it by continent → take the lifeExp column → mean per group.*

`.mean()` is swappable: `.max()`, `.min()`, `.count()`, `.median()`, `.sum()` ...

In [ ]:
# How many countries per continent? (count of rows per group)
print(gm2007.groupby("continent")["country"].count())



In [ ]:
# Total world population by continent, in billions:
print(gm2007.groupby("continent")["pop"].sum() / 1_000_000_000)

In [ ]:
# Several statistics at once: .agg() with a list
gm2007.groupby("continent")["lifeExp"].agg(["mean", "min", "max"])

In [ ]:
# You can group by SEVERAL columns - e.g. continent AND year, across the whole data.
# .reset_index() turns the result back into a regular DataFrame (handy for plotting!):
trends = gapminder.groupby(["continent", "year"])["lifeExp"].mean().reset_index()

trends.head(8)     # Africa's average, wave by wave... then the Americas, and so on

# Combining datasets: pd.merge

Real projects almost never live in ONE table. Your survey sits in one file, the country indicators in another - and the analysis needs both. `pd.merge()` glues two DataFrames together on a shared column (the *key*):

```python
combined = pd.merge(left_table, right_table, on="key_column")
```

Let's attach our little Nordic table from Day 2 (capitals, EU membership) to the 2007 Gapminder data:

In [ ]:
# A second table: the Nordic countries file from Day 2
nordics = pd.read_csv(DATA_URL + "nordic_countries.csv")
nordics

In [ ]:
gm2007.head()

In [ ]:
# Merge on the shared column "country":
merged = pd.merge(gm2007, nordics, on="country")

print(merged.shape)   # only 5 rows survived!


In [ ]:
merged.head()

In [ ]:
merged[["country", "capital", "lifeExp", "gdpPercap", "eu_member"]]

Only 5 rows! By default `merge` keeps **only the rows that appear in BOTH tables** (an *inner* join) - the other 137 countries had no partner in the Nordic table and were dropped.

If you would rather KEEP everything from one side, say `how="left"`:

| `how=` | Rows kept | Unmatched rows |
|---|---|---|
| `inner` *(default)* | Only keys present in **both** tables | Dropped |
| `left` | Every row of the **left** table | Kept, right-side columns = `NaN` |
| `right` | Every row of the **right** table | Kept, left-side columns = `NaN` |
| `outer` | Every key from **either** table | Kept, `NaN` on the missing side |



In [ ]:
gm2007.head(2)

In [ ]:
nordics.head(2)

In [ ]:
# how="left": keep ALL rows of the left table (gm2007), and fill the
# right table's columns with NaN wherever there is no match:
merged_all = pd.merge(gm2007, nordics, on="country", how="left")

print(merged_all.shape)   
print(gm2007.shape)                                   # all 142 rows kept
                                # all 142 rows kept
merged_all.head(10)

One warning: merge matches names **exactly**. 

To pandas, "USA" and "United States" are not the same - and in an inner join, a mismatched name disappears *silently*. 

In [ ]:
# The classic trap: a G7 table where one country name does not match.
g7 = pd.DataFrame({
    "country": ["Canada", "France", "Germany", "Italy", "Japan", "United Kingdom", "USA"],
    "g7_member": [True, True, True, True, True, True, True],
})

g7

In [ ]:
gm2007.tail(10)

In [ ]:
g7_data = pd.merge(gm2007, g7, on="country")
print(len(g7_data))                  # 6 rows - but the G7 has SEVEN members!
print(g7_data["country"].tolist())   # who is missing... and why?

The USA vanished without a word: gapminder spells it `"United States"`, so the inner join dropped the stranger. **Always check the row count after a merge.** The fix - harmonize the name, merge again:

In [ ]:
g7.head(1)

In [ ]:
# Fix the name in the g7 table, then merge again:
#.loc[which rows, which column]
#so the row here is where the country is USA
#the column is the country column
#we want to rename it to United States
g7.loc[g7["country"] == "USA", "country"] = "United States"



In [ ]:


g7_data = pd.merge(gm2007, g7, on="country")
print(len(g7_data))                  # 7 - everyone on board

# And the payoff of every merge - questions across BOTH tables:
print(f"G7 mean life expectancy:    {g7_data['lifeExp'].mean():.1f} years")
print(f"World mean life expectancy: {gm2007['lifeExp'].mean():.1f} years")

# Dates: pulling out the parts you need

One more everyday skill before the charts. Dates usually arrive as **text** - Python cannot calculate with "2021-09-26" until we tell it this is a date. That is the job of `pd.to_datetime()`; afterwards, the `.dt` accessor unlocks every part of the date.

Our example: turnout in German federal elections *(official figures from the Federal Returning Officer, [bundeswahlleiterin.de](https://www.bundeswahlleiterin.de))*.

In [ ]:
# Load the elections file and inspect the types:
elections = pd.read_csv(DATA_URL + "german_elections.csv")

print(elections.dtypes)        # election_date is 'object' - plain text so far!
elections

In [ ]:
# Convert the text column into real dates:
elections["election_date"] = pd.to_datetime(elections["election_date"])

print(elections.dtypes)        # datetime64 - pandas now UNDERSTANDS these values

In [ ]:
elections.head(2)

In [ ]:
# The .dt accessor extracts any part of a date - most often you want the year:
elections["year"] = elections["election_date"].dt.year
elections["month"] = elections["election_date"].dt.month

elections
# (There is also .dt.day, .dt.day_name(), and more - always the same pattern.)

In [ ]:
#you can alternatively do the same with strings
# 1. datetime → string, e.g. "2024-11-05"
elections["date_str"] = elections["election_date"].dt.strftime("%Y-%m-%d")

#"string format time" - Year month day

elections.head(2)

In [ ]:
# 2. same extractions, now as string slicing
elections["year"]  = elections["date_str"].str[:4]    # "2024"
elections["month"] = elections["date_str"].str[5:7]   # "11"
elections.head(2)

| Code | Meaning | Example |
|---|---|---|
| `%Y` | 4-digit year | `2024` |
| `%y` | 2-digit year | `24` |
| `%m` | month, zero-padded | `11` |
| `%B` / `%b` | month name / abbreviated | `November` / `Nov` |
| `%d` | day, zero-padded | `05` |
| `%A` / `%a` | weekday name / abbreviated | `Tuesday` / `Tue` |

# matplotlib: plotting!

**matplotlib** is Python's foundational plotting library - nearly every other plotting tool (including seaborn) builds on it. The pattern is always:

```python
plt.figure(...)      # 1. create a canvas (optional but nice for sizing)
plt.plot(x, y)       # 2. draw something - plot/bar/hist/scatter...
plt.title(...)       # 3. label it (ALWAYS label your charts!)
plt.show()           # 4. display it
```



In [ ]:
# The data for our line: Germany, all 12 waves
germany = gapminder[gapminder["country"] == "Germany"]

germany.head(2)

**Line Chart**

In [ ]:
plt.figure(figsize=(9, 5))                    # canvas: 9 x 5 inches
plt.plot(germany["year"], germany["lifeExp"]) # line chart: x = year, y = life expectancy

# lets label the chart
plt.title("Life expectancy in Germany, 1952-2007")
plt.xlabel("Year")
plt.ylabel("Life expectancy (years)")

plt.show()

In [ ]:
# Style the line & add more countries - each plt.plot() adds a line.
# label= ... + plt.legend() tells readers which line is which:
for country, color in [("Germany", "black"), ("Vietnam", "red"), ("Rwanda", "blue")]: #for each of these countries and corresponding color
    subset = gapminder[gapminder["country"] == country]
    # keep only this country's rows, so plt.plot draws one line for one country
    print(subset.head(3))
    plt.plot(subset["year"], subset["lifeExp"],
    #ok we want the x and y on different access for this specifc subset
             color=color, marker="o", linewidth=2, label=country)
plt.title("Three very different paths, 1952-2007")
plt.xlabel("Year")
plt.ylabel("Life expectancy (years)")
plt.legend()                                   # uses the label= of each line
plt.grid(alpha=0.2)                            # subtle guide lines (alpha = transparency)
plt.show()



In [ ]:
# Shortcut: every DataFrame can plot ITSELF - pandas calls matplotlib for you.
# Great for quick looks (label with title=, then polish in matplotlib when publishing):
germany.plot(x="year", y="gdpPercap", figsize=(8, 4), marker="o",
             title="Germany: GDP per capita, 1952-2007", legend=False)
plt.show()

**Bar chart**

In [ ]:
# Bar charts compare categories. Horizontal (barh) fits country names best.
top10 = gm2007.sort_values("gdpPercap", ascending=False).head(10)
top10

In [ ]:


plt.figure(figsize=(9, 5)) #how big the chart is
plt.barh(top10["country"], top10["gdpPercap"], color="tab:green")
# we want a horizontal bar chart 
plt.title("Top 10 GDP per capita, 2007")
plt.xlabel("GDP per capita (dollars)")
#here we invert axis
plt.gca().invert_yaxis()      # richest on TOP (barh plots bottom-up by default)
#plt.gca() #get current axi
#.invert_yaxis()  inverts the order of the axis
plt.show()

**Histogram**

In [ ]:
# Histograms show a DISTRIBUTION: how are values spread out?
plt.figure(figsize=(9, 5))
plt.hist(gm2007["lifeExp"], bins=10, color="purple", edgecolor="white")

plt.title("Distribution of life expectancy across 142 countries, 2007")
plt.xlabel("Life expectancy (years)")
plt.ylabel("Number of countries")
plt.show()



**Scatterplot**

In [ ]:
# Scatter plots show RELATIONSHIPS between two variables.
# The most famous one in development research: wealth vs. health.
plt.figure(figsize=(9, 5))
plt.scatter(gm2007["gdpPercap"], gm2007["lifeExp"], alpha=0.6)

plt.title("Wealth and health, 2007")
plt.xlabel("GDP per capita (dollars)")
plt.ylabel("Life expectancy (years)")
plt.xscale("log")     # log scale! GDP is extremely skewed (yesterday's np.log, as an axis)
plt.show()

In [ ]:
gm2007.head()

In [ ]:
gm2007.plot.scatter(x="gdpPercap", y="lifeExp",
                      s=gm2007["pop"] / 1e6,   # bubble size = population
                      alpha=0.5,
                       logx=True,       # log income axis
                      title="Income vs life expectancy, 2007")


# seaborn: statistical plots with style

**seaborn** (`sns`) sits on top of matplotlib and speaks fluent DataFrame: you name the columns, it handles colors, legends, and statistics. One line of setup makes *all* your plots prettier:

In [ ]:
# Activate seaborn's clean look (affects all plots from now on - even matplotlib ones!)
sns.set_theme(style="whitegrid")

# The scatter from before - now with continents as colors (hue=) and population as size:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=gm2007,
 x="gdpPercap", y="lifeExp",
                hue="continent",         # color the dots by continent
                size="pop",              # scale the dots by population
                sizes=(20, 800),         # smallest & biggest dot size
                alpha=0.7)

plt.xscale("log")
plt.title("Wealth & health, 2007 - by continent and population")
plt.xlabel("GDP per capita (dollars, log scale)")
plt.ylabel("Life expectancy (years)")
plt.legend(bbox_to_anchor=(1.02, 1))     # move the legend outside the plot
plt.show()

# One seaborn line replaced a LOT of manual matplotlib work.

In [ ]:
# Boxplots: compare a distribution ACROSS groups - the visual .groupby()!
plt.figure(figsize=(9, 5))
sns.boxplot(data=gm2007, x="continent", y="lifeExp", hue="continent", palette="Set2")

plt.title("Life expectancy by continent, 2007")
plt.xlabel("")
plt.ylabel("Life expectancy (years)")
plt.show()

# How to read a box: the line in the middle = median, the box = middle 50% of
# countries, the whiskers = the typical range, lone dots = outliers.

In [ ]:

gapminder.head()

In [ ]:
# Remember our groupby 'trends' table? seaborn turns it straight into a story:

#i want to group by continent and year and get the mean of life expectance
trends = gapminder.groupby(["continent", "year"])["lifeExp"].mean().reset_index()
trends.head()


In [ ]:
plt.figure(figsize=(10, 6)) #size of the chart
#i want to plot a line chart where i have year on x axis and life expectance on y axis
sns.lineplot(data=trends, x="year", y="lifeExp", hue="continent", #different colors for each continent (i like these colors)
             linewidth=2.5, marker="o")
#i want to title the chart
plt.title("Average life expectancy by continent, 1952-2007")
#i want to label the x axis
plt.xlabel("Year")
#i want to label the y axis
plt.ylabel("Life expectancy (years)")
#i want to show the chart
plt.show()



# Welcome abroad Python Users!

![Celebrate](https://media.giphy.com/media/dUNNnf1vwvhRuGCBMb/giphy.gif)

You will recreate, **from scratch**, one of the most famous data visualizations ever made: **Hans Rosling's animated bubble chart** - the one from [the BBC clip](https://www.youtube.com/watch?v=jbkSRLYSojo) where 200 years of world history play out in bubbles.

For this we bring out **plotly express** (`px`): interactive charts (hover, zoom!) with animation built in.

In [ ]:
gapminder.head() #lets remind ourselves what the data looks like

In [ ]:
# THE Hans Rosling chart - every argument is a column name doing a job:
fig = px.scatter( #we call the scatter function from plotly express
    gapminder, #the data we want to plot
    x="gdpPercap",            # wealth on the x-axis...
    y="lifeExp",              # ...health on the y-axis
    size="pop",               # bubble size = population
    color="continent",        # bubble color = continent
    hover_name="country",     # what you see when you hover a bubble
    animation_frame="year",   # one animation frame per year!
    log_x=True,               # log scale for skewed GDP
    size_max=60, #the maximum size of the bubbles
    range_y=[25, 90], #the range of the y axis
    title="The world getting healthier and richer, 1952-2007 (press Play!)",
    labels={"gdpPercap": "GDP per capita (dollars)", "lifeExp": "Life expectancy (years)"}, #the labels of the x and y axis
)
fig.show()      #we show the chart

# Press play. Watch Asia surge, see China's famine dip around 1960,
# Rwanda plunge in the 1990s... Then HOVER over bubbles - it's all interactive!

## World maps (choropleths)

Remember the mysterious `iso_alpha` column from this morning? Standardized 3-letter country codes - and the key to **maps**. A *choropleth* colors each country by a value:

In [ ]:
gm2007.head()

In [ ]:
# A world map of life expectancy in 2007 - hover over any country!
fig = px.choropleth( #we call the choropleth function from plotly express
    gm2007, #the data we want to plot
    locations="iso_alpha",              # the ISO codes tell plotly WHERE each country is
    color="lifeExp",                    # what the colors mean
    hover_name="country",
    color_continuous_scale="Viridis",
    title="Life expectancy around the world, 2007",
)
fig.show()

In [ ]:
# And because we can: the SAME map, animated through time. Two extra arguments!
fig = px.choropleth(
    gapminder,
    locations="iso_alpha", #he library ships with its own world base map
    color="lifeExp",
    hover_name="country",
    animation_frame="year",             # again - any px chart can animate
    color_continuous_scale="Viridis",
    range_color=[30, 85],               # fixed scale so colors are comparable across years
    title="Life expectancy, 1952-2007 (press Play!)",
)
fig.show()

In [ ]:
# Last showpiece: a sunburst - click a continent to zoom in, click the center to zoom out!
fig = px.sunburst(
    gm2007,
    path=["continent", "country"],      # the hierarchy: rings from inside out
    values="pop",                       # slice size = population
    title="World population 2007 - click to explore!",
)
fig.show()

# Putting it ALL together

One final cell that uses **everything from three days**: loading, filtering, groupby, new columns, loops, f-strings - and a 2×2 dashboard of the world in 2007. Read it top to bottom and enjoy how much you understand now.

In [ ]:
# ============= THE WHOLE COURSE IN ONE CELL =============

# --- Data prep (Day 3 morning) ---
gm2007 = gapminder[gapminder["year"] == 2007].copy()
gm2007["pop_millions"] = gm2007["pop"] / 1_000_000          # new column

# --- A 2x2 grid of plots: fig is the canvas, axes a 2x2 "list of lists" of panels ---
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("The World in 2007 - GESIS Python Course Finale", fontsize=15, fontweight="bold")

# Panel 1 (top-left): the continents' long-run trends (groupby + loop!)
trends = gapminder.groupby(["continent", "year"])["lifeExp"].mean().reset_index()
for continent in trends["continent"].unique():
    subset = trends[trends["continent"] == continent]
    axes[0, 0].plot(subset["year"], subset["lifeExp"], marker="o", label=continent)
axes[0, 0].set_title("Life expectancy by continent")
axes[0, 0].legend(fontsize=8)

# Panel 2 (top-right): distribution of life expectancy (histogram)
axes[0, 1].hist(gm2007["lifeExp"], bins=20, color="tab:purple", edgecolor="white")
axes[0, 1].set_title("Distribution of life expectancy")

# Panel 3 (bottom-left): the 10 most populous countries (sort + barh)
top_pop = gm2007.sort_values("pop_millions", ascending=False).head(10)
axes[1, 0].barh(top_pop["country"], top_pop["pop_millions"], color="tab:green")
axes[1, 0].invert_yaxis()
axes[1, 0].set_title("Most populous countries (millions)")

# Panel 4 (bottom-right): wealth vs health scatter
axes[1, 1].scatter(gm2007["gdpPercap"], gm2007["lifeExp"], alpha=0.6)
axes[1, 1].set_xscale("log")
axes[1, 1].set_title("Wealth vs. health (log GDP)")

plt.tight_layout()
plt.show()


# Wrap-up: three days, one new superpower

Look at what you can do now - **all of it from zero**:

| Day | You learned |
|-----|-------------|
| **1** | variables · data types · operations · strings & f-strings · lists · tuples · dictionaries · sets · importing libraries |
| **2** | if/elif/else · for & while loops · list comprehensions · writing functions · reading & writing files · CSVs |
| **3** | pandas: loading, inspecting, filtering, sorting, new columns, missing values · numpy · groupby · merge · dates · matplotlib · seaborn · plotly |

## Where to go next

**Practice (the only thing that really works):**
- Load a dataset **from your own research** into Colab and explore it with your new tools - this week, while it's fresh!
- [Gapminder's site](https://www.gapminder.org/data/) has hundreds more indicators to play with.

**Reference & learning:**
- [pandas cheat sheet (official, 2 pages)](https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf) - print it, love it
- *Python for Data Analysis* by Wes McKinney (the creator of pandas) - free online: https://wesmckinney.com/book/
- [Google's free Python class](https://developers.google.com/edu/python)
- [The Python Graph Gallery](https://python-graph-gallery.com/) - every chart type, with copy-paste code

**When you are stuck** *(everyone is, daily)*: read the **last line** of the error first · search the error message online · [Stack Overflow](https://stackoverflow.com) almost always has your answer · AI assistants can be helpful - but always **run and understand** what they give you.

**And after this course:** you are now ready for courses on **text analysis, web scraping, and machine learning** - keep an eye on the [GESIS training program](https://training.gesis.org).

## Thank you!

It was a joy teaching you. Questions later? Write me: **a.elshehawy@ucl.ac.uk**

*Now: one last exercise block.*